# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

https://colab.research.google.com/github/jasontpv/DS2002FA26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb#scrollTo=0lkjZf6fyZOK

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [2]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # Loop up to `tries` times
    for attempt in range(tries):
      try:
        # Request with a timeout
        r = requests.get(BASE, params=params, timeout = 10)

        # Return pd.DataFrame(r.json()['daily']) on 200
        if r.status_code == 200:
          return pd.DataFrame(r.json()['daily'])

        # Raise on a status that is not in RETRY_STATUSES
        elif r.status_code not in RETRY_STATUSES:
          r.raise_for_status()
          raise RuntimeError(f'Unexpected status code of {r.status_code}')

      except requests.exceptions.RequestException:
        # If last attempt
        if attempt == tries - 1:
          raise

      # Otherwise sleep (increasing each time) and try again
      if attempt < tries - 1:
        time.sleep(2 ** attempt)

    # Raise if every attempt failed
    raise RuntimeError('All attempts to request weather failed')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [3]:
used_fallback = False

try:
  # Call get_weather once for a range that spans both dates
  weather = get_weather('2026-09-05', '2026-10-03')

  # Filter two dates
  weather = weather[weather['time'].isin(['2026-09-05', '2026-10-03'])]
  # Reset row numbers
  weather = weather.reset_index(drop=True)
except:
  # If call fails, use fallback and record
  used_fallback = True
  print('Weather API failed. Using fallback')

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [4]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# If used_fallback == True, build `wx` from FALLBACK
if used_fallback:
  wx = FALLBACK.copy()
  # Add a 'source' column to `wx` -- 'fallback sample'
  wx['source'] = 'fallback sample'
else:
  # Use real API data
  wx = weather.copy()
  # Add a 'source' column to `wx` -- 'open-meteo'
  wx['source'] = 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [9]:
# Produce a frame and rename columns
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})

# Convert date column from strings to datetime objects
wx['date'] = pd.to_datetime(wx['date'])

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [12]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [15]:
# Row with highest percipitation
highest_perc = wx.loc[wx['precip_mm'].idxmax()]

# Row with lowest percipitation
lowest_perc = wx.loc[wx['precip_mm'].idxmin()] # Index

# Find the difference in percipitation
difference = highest_perc['precip_mm'] - lowest_perc['precip_mm']

# Print sentence for vendor
print(f'The date with the highest percipitation was {highest_perc['date'].strftime('%Y-%m-%d')}.')
print(f'The date with the lowest percipitation was {lowest_perc['date'].strftime('%Y-%m-%d')}.')
print(f'The difference in percipitation was {difference}.')

The date with the highest percipitation was 2026-10-03.
The date with the lowest percipitation was 2026-09-05.
The difference in percipitation was 12.2.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [27]:
from datetime import datetime

# Date the information was pulled
date_pulled = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Print the endpoint
print(f'Endpoint: {BASE}')

# Print the coordinates
print('Coordinates: latitude = 38.03, longitude = -78.51')

# Print the date range
print('Date range: 2026-09-05 to 2026-10-03')

print(f'Date pulled: {date_pulled}')

# Print whether the data is real or fallback
if used_fallback:
  print('Data is fallback, not real')
else:
  print('Data is real, not fallback')

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude = 38.03, longitude = -78.51
Date range: 2026-09-05 to 2026-10-03
Date pulled: 2026-10-09 18:26:26
Data is real, not fallback


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

Answer: If get_weather function returned an empty DataFrame on failure instead of raising an exception, merging the 200-row sales DataFrame with the empty DataFrame using a left join would keep all the sales data but the weather columns would contain NaN. Therefore, the chart would have no valid rain data to compare against revenue. Because the merge ran with no errors, nobody may catch the problem. However, this table does not contain all the information that you want, potentially leading to incorrect conclusions.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [29]:
# Pull weather information for Charlottesville
charlottesville = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
charlottesville = charlottesville[charlottesville['time'].isin(['2026-09-05', '2026-10-03'])].copy()

# Pull weather information for Orlando
orlando = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
orlando = orlando[orlando['time'].isin(['2026-09-05', '2026-10-03'])].copy()

# Add city column
charlottesville['city'] = 'Charlottesville'
orlando['city'] = 'Orlando'

# Combine into one long-format frame
wx_combined = pd.concat(
    [charlottesville, orlando],
    ignore_index=True
)

# Normalize column names
wx_combined = wx_combined.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})

# Convert date column from strings to datetime objects
wx_combined['date'] = pd.to_datetime(wx_combined['date'])

# Reorder the columns
wx_combined = wx_combined[['city', 'date', 'temp_max', 'precip_mm']]

print(wx_combined)

              city       date  temp_max  precip_mm
0  Charlottesville 2026-09-05      32.1        3.9
1  Charlottesville 2026-10-03      30.4       14.5
2          Orlando 2026-09-05      32.1        3.9
3          Orlando 2026-10-03      30.4       14.5
